# 💻 Práctica 8: Generación de imágenes y reconstrucción

En esta práctica vamos a explorar los fundamentos de los **modelos generativos** aplicados a imágenes.

Trabajaremos con el dataset **Fashion-MNIST**, que contiene imágenes en escala de grises de 10 categorías de ropa y accesorios (remeras, zapatillas, bolsos, etc.).

A lo largo de la práctica vamos a:
- Implementar un **Autoencoder (AE)** clásico y explorar su espacio latente.
- Extenderlo a un **Variational Autoencoder (VAE)** y ver cómo cambia la estructura del espacio latente.
- Usar ambos modelos para **denoising** e **interpolación** entre imágenes.

📌 Antes de comenzar, generá una copia de este archivo en tu Drive y trabajá en esa copia.

---

## 🧭 Parte 1: Práctica guiada

En esta primera parte vamos a construir un pipeline completo que incluye:

- Carga y visualización del dataset Fashion-MNIST.
- Implementación y entrenamiento de un Autoencoder (AE).
- Visualización y análisis del espacio latente.
- Extensión al Variational Autoencoder (VAE).
- Aplicaciones: denoising e interpolación en el espacio latente.

### Setup inicial

Importamos las bibliotecas necesarias:

In [ ]:
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE

Seleccionamos el dispositivo de ejecución, prefiriendo la GPU si está disponible:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo a utilizar:", device)

### Dataset Fashion-MNIST

**Fashion-MNIST** es un dataset de imágenes en escala de grises de 28x28 píxeles con 10 clases:

| Clase | Categoría |
|---|---|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

Tiene 60.000 imágenes de entrenamiento y 10.000 de test. Se descarga automáticamente desde torchvision.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset,  batch_size=128, shuffle=False)

print(f"Train: {len(train_dataset)} imagenes")
print(f"Test: {len(test_dataset)} imagenes")

Visualizamos algunas imágenes del dataset:

In [ ]:
class_names = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for cls in range(10):
    idxs = [i for i, (_, label) in enumerate(train_dataset) if label == cls]
    for row in range(2):
        idx = random.choice(idxs)
        img, label = train_dataset[idx]
        axes[row, cls].imshow(img.squeeze(), cmap="gray")
        axes[row, cls].axis("off")
        if row == 0:
            axes[row, cls].set_title(class_names[label], fontsize=7)

plt.suptitle("Muestras por clase - Fashion-MNIST", y=1.02)
plt.tight_layout()
plt.show()

❓ Preguntas:

1. ¿Qué diferencias visuales notás entre las distintas categorías?
2. ¿Cuáles te parecen más difíciles de distinguir entre sí? ¿Por qué?

### Autoencoder (AE)

Un Autoencoder aprende a comprimir una imagen a un vector de baja dimensión (**espacio latente**) y luego reconstruirla.

La arquitectura tiene dos partes:
- **Encoder**: imagen → vector `z` (cuello de botella)
- **Decoder**: vector `z` → imagen reconstruida

El modelo se entrena minimizando el error de reconstrucción píxel a píxel (MSE).

No necesita etiquetas: la supervisión viene de los propios datos.

#### Arquitectura del AE

El **Encoder** toma una imagen de 28x28, aplica dos convoluciones con stride 2 (que reducen la resolución espacial a la mitad en cada paso) y proyecta a un vector de dimensión `latent_dim`:

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),  # -> 32x14x14
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # -> 64x7x7
            nn.ReLU()
        )
        self.fc = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)  # aplanar
        return self.fc(x)

El **Decoder** realiza la operación inversa: proyecta el vector latente de vuelta al espacio de imágenes usando `ConvTranspose2d`, que es la operación simétrica a `Conv2d` con stride (que aumenta la resolución espacial):

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 64 * 7 * 7)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1), # -> 32x14x14
            nn.ReLU(),
            nn.ConvTranspose2d(32,  1, kernel_size=3, stride=2, padding=1, output_padding=1), # -> 1x28x28
            nn.Sigmoid()  # valores en [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 64, 7, 7)  # reorganizar a volumen 3D
        return self.deconv(x)

Juntamos ambas partes en el Autoencoder completo:

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

Verificamos que el modelo produzca una salida de la misma forma que la entrada:

In [ ]:
ae = Autoencoder(latent_dim=16).to(device)

x_test = torch.randn(4, 1, 28, 28).to(device)
out = ae(x_test)

print("Input shape:", x_test.shape)
print("Output shape:", out.shape)

#### Entrenamiento del AE

El loop de entrenamiento es idéntico al de cualquier red supervisada, con la diferencia de que la etiqueta es la propia imagen de entrada:

In [ ]:
def train_ae(model, loader, optimizer, device, num_epochs=15):
    model.train()
    losses = []

    for epoch in range(num_epochs):
        epoch_loss = 0.0

        for images, _ in loader:  # las etiquetas no se usan
            images = images.to(device)

            optimizer.zero_grad()
            recon = model(images)
            loss = F.mse_loss(recon, images)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}] | Loss: {avg_loss:.4f}")

    return losses

In [ ]:
ae = Autoencoder(latent_dim=16).to(device)
optimizer_ae = optim.Adam(ae.parameters(), lr=1e-3)

ae_losses = train_ae(ae, train_loader, optimizer_ae, device, num_epochs=15)

In [ ]:
plt.plot(ae_losses)
plt.xlabel("Epoca")
plt.ylabel("MSE Loss")
plt.title("Curva de entrenamiento - Autoencoder")
plt.grid(True)
plt.show()

#### Visualización de reconstrucciones

Comparamos imágenes originales con sus reconstrucciones:

In [ ]:
def show_reconstructions(model, dataset, n=8):
    model.eval()
    idxs = random.sample(range(len(dataset)), n)
    images = torch.stack([dataset[i][0] for i in idxs]).to(device)

    with torch.no_grad():
        recons = model(images)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3))
    for i in range(n):
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap="gray")
        axes[0, i].axis("off")
        axes[1, i].imshow(recons[i].cpu().squeeze(), cmap="gray")
        axes[1, i].axis("off")

    axes[0, 0].set_ylabel("Original", fontsize=9)
    axes[1, 0].set_ylabel("Reconstruccion", fontsize=9)

    plt.suptitle("Originales vs. Reconstrucciones")
    plt.tight_layout()
    plt.show()

show_reconstructions(ae, test_dataset)

❓ Preguntas:

1. ¿Qué detalles se pierden en la reconstrucción? ¿Cuáles se preservan?
2. ¿Qué clases se reconstruyen mejor y cuáles peor? ¿A qué se puede deber?
3. ¿Qué pasaría si aumentaras `latent_dim` a 128? ¿Y si lo reducís a 2?

#### Visualización del espacio latente

Extraemos los vectores latentes del conjunto de test y los proyectamos en 2D con t-SNE para visualizar si el espacio latente agrupa las clases de forma significativa, a pesar de que el modelo nunca vio las etiquetas durante el entrenamiento:

In [ ]:
def extract_latents(encoder, loader, device, max_batches=20):
    encoder.eval()
    all_z, all_labels = [], []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(loader):
            if batch_idx >= max_batches:
                break
            images = images.to(device)
            z = encoder(images)
            all_z.append(z.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_z, axis=0), np.concatenate(all_labels, axis=0)

ae_latents, ae_labels = extract_latents(ae.encoder, test_loader, device)
print(f"Latent shape: {ae_latents.shape}")

In [ ]:
def plot_latent_space(latents, labels, title="Espacio latente (t-SNE)"):
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    z_2d = tsne.fit_transform(latents)

    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1], c=labels, cmap="tab10", s=5, alpha=0.7)
    cbar = plt.colorbar(scatter, ticks=range(10))
    cbar.set_ticklabels(class_names)

    plt.title(title)
    plt.xlabel("t-SNE dim 1")
    plt.ylabel("t-SNE dim 2")
    plt.tight_layout()
    plt.show()

plot_latent_space(ae_latents, ae_labels, title="Espacio latente - AE (t-SNE)")

❓ Preguntas:

1. ¿Se forman grupos separados por clase? ¿Cuáles quedan más juntos y por qué?
2. ¿Qué esperarías ver si tomás un punto del espacio latente que cae entre dos grupos y lo decodificás?
3. ¿El espacio latente del AE parece continuo?

### Variational Autoencoder (VAE)

El VAE extiende el AE haciendo que el espacio latente sea **probabilístico**:
- El encoder no produce un vector `z` fijo, sino una **distribución**: media `mu` y log-varianza `log_var`.
- Se samplea `z ~ N(mu, sigma^2)` usando el **reparametrización**: `z = mu + sigma * epsilon`, con `epsilon ~ N(0,1)`.
- La función de pérdida agrega un término de **KL divergence** que empuja al espacio latente a parecerse a N(0,1).

Esto garantiza que el espacio latente sea continuo y navegable: podemos samplear cualquier punto de N(0,1) y decodificarlo en una imagen coherente.

#### Arquitectura del VAE

El encoder del VAE produce dos vectores en lugar de uno: `mu` y `log_var`:

In [ ]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)
        self.fc_log_var = nn.Linear(64 * 7 * 7, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)

        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)

        return mu, log_var

La **reparametrización** permite que el gradiente fluya a través del sampleo:
- En lugar de samplear `z ~ N(mu, sigma^2)` directamente (no diferenciable),
- Calculamos `z = mu + sigma * epsilon` donde `epsilon ~ N(0,1)` es ruido externo.
- El gradiente fluye por `mu` y `sigma`; la aleatoriedad queda aislada en `epsilon`.

In [ ]:
def reparametrize(mu, log_var):
    std = torch.exp(0.5 * log_var)
    epsilon = torch.randn_like(std)
    return mu + std * epsilon

El decoder del VAE es idéntico al del AE (reutilizamos la clase `Decoder`).

Armamos el VAE completo:

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = reparametrize(mu, log_var)
        recon = self.decoder(z)
        return recon, mu, log_var

    def generate(self, n, device):
        with torch.no_grad():
            z = torch.randn(n, self.encoder.fc_mu.out_features).to(device)
            return self.decoder(z)

#### Función de pérdida del VAE

La pérdida del VAE combina dos términos:
- **Reconstrucción**: MSE entre imagen original y reconstruida (igual que el AE).
- **KL divergence**: penaliza que la distribución aprendida `q(z|x)` se aleje de N(0,1). Tiene forma cerrada para N(0,1):

$$D_{KL} = -\frac{1}{2} \sum_{j} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2 \right)$$

El parámetro `beta` controla el peso relativo del término KL.

In [ ]:
def vae_loss(recon, x, mu, log_var, beta=1.0):
    # Perdida de reconstruccion
    recon_loss = F.mse_loss(recon, x, reduction="sum") / x.size(0)

    # KL divergence con forma cerrada
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)

    return recon_loss + beta * kl_loss, recon_loss, kl_loss

#### Entrenamiento del VAE

In [ ]:
def train_vae(model, loader, optimizer, device, num_epochs=15, beta=1.0):
    model.train()
    total_losses, recon_losses, kl_losses = [], [], []

    for epoch in range(num_epochs):
        epoch_total, epoch_recon, epoch_kl = 0.0, 0.0, 0.0

        for images, _ in loader:
            images = images.to(device)
            optimizer.zero_grad()

            recon, mu, log_var = model(images)
            loss, recon_loss, kl_loss = vae_loss(recon, images, mu, log_var, beta)

            loss.backward()
            optimizer.step()

            epoch_total += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()

        n = len(loader)
        total_losses.append(epoch_total / n)
        recon_losses.append(epoch_recon / n)
        kl_losses.append(epoch_kl / n)

        print(
            f"Epoch [{epoch+1}/{num_epochs}] | "
            f"Loss: {epoch_total/n:.2f} | "
            f"Recon: {epoch_recon/n:.2f} | "
            f"KL: {epoch_kl/n:.2f}"
        )

    return total_losses, recon_losses, kl_losses

In [ ]:
vae = VAE(latent_dim=16).to(device)
optimizer_vae = optim.Adam(vae.parameters(), lr=1e-3)

total_losses, recon_losses, kl_losses = train_vae(
    vae, train_loader, optimizer_vae, device, num_epochs=15, beta=1.0
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, losses, title in zip(
    axes,
    [total_losses, recon_losses, kl_losses],
    ["Loss total", "Reconstruccion", "KL divergence"]
):
    ax.plot(losses)
    ax.set_title(title)
    ax.set_xlabel("Epoca")
    ax.grid(True)

plt.suptitle("Curvas de entrenamiento - VAE")
plt.tight_layout()
plt.show()

❓ Preguntas:

1. ¿Qué ocurre con el término KL a lo largo del entrenamiento? ¿Por qué?
2. ¿Qué pasaría si `beta = 0`? ¿En qué modelo se convierte el VAE?

#### Comparación de espacios latentes: AE vs VAE

Vamos a extraer y visualizar los espacios latentes de ambos modelos  para compararlos.

Para el VAE usamos la media (`mu`) como representación latente:

In [ ]:
def extract_latents_vae(model, loader, device, max_batches=20):
    model.eval()
    all_mu, all_labels = [], []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(loader):
            if batch_idx >= max_batches:
                break
            images = images.to(device)
            mu, _ = model.encoder(images)
            all_mu.append(mu.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_mu, axis=0), np.concatenate(all_labels, axis=0)

vae_latents, vae_labels = extract_latents_vae(vae, test_loader, device)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, latents, labels, title in zip(
    axes,
    [ae_latents, vae_latents],
    [ae_labels, vae_labels],
    ["Espacio latente - AE", "Espacio latente - VAE"]
):
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    z_2d = tsne.fit_transform(latents)
    sc = ax.scatter(z_2d[:, 0], z_2d[:, 1], c=labels, cmap="tab10", s=5, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("dim 1")
    ax.set_ylabel("dim 2")

cbar = plt.colorbar(sc, ax=axes[1], ticks=range(10))
cbar.set_ticklabels(class_names)

plt.suptitle("Comparacion de espacios latentes (t-SNE)")
plt.tight_layout()
plt.show()

❓ Preguntas:

1. ¿Qué diferencias observás entre el espacio latente del AE y el del VAE?
2. ¿En cuál de los dos el espacio latente parece más continuo? ¿Por qué?
3. ¿Qué ventaja tiene eso para la generación de nuevas imágenes?

### Aplicaciones

#### Aplicación 1: Denoising con el AE

Una aplicación natural del AE es hacer **denoising**: si le pasamos una imagen ruidosa, el encoder extrae la estructura esencial (ignorando el ruido) y el decoder reconstruye una versión limpia.

Para probarlo, vamos a corromper imágenes con ruido gaussiano y ver qué tan bien las recupera el AE.

Tengamos en cuenta que para este caso el AE fue entrenado únicamente con imánes limpias.

In [ ]:
def add_noise(images, noise_factor=0.4):
    noise = torch.randn_like(images) * noise_factor
    return torch.clamp(images + noise, 0.0, 1.0)

def show_denoising(model, dataset, n=8, noise_factor=0.4, title="Denoising"):
    model.eval()
    idxs = random.sample(range(len(dataset)), n)
    images = torch.stack([dataset[i][0] for i in idxs]).to(device)
    noisy = add_noise(images, noise_factor)

    with torch.no_grad():
        recons = model(noisy)

    fig, axes = plt.subplots(3, n, figsize=(n * 1.5, 4.5))
    row_labels = ["Original", f"Ruidosa (factor={noise_factor})", "Reconstruida"]

    for i, (orig, noisy_img, recon) in enumerate(zip(images, noisy, recons)):
        for row, img in enumerate([orig, noisy_img, recon]):
            axes[row, i].imshow(img.cpu().squeeze(), cmap="gray")
            axes[row, i].axis("off")

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=9)

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

show_denoising(ae, test_dataset, noise_factor=0.4, title="Denoising con AE")

❓ Preguntas:

1. ¿Por qué el AE puede hacer denoising si no fue entrenado con imágenes ruidosas?
2. Probá con `noise_factor=0.8`. ¿Qué pasa con la calidad de la reconstrucción?
3. ¿En qué casos el denoising falla? ¿Qué información se pierde?

#### Aplicación 2: Interpolación en el espacio latente con el VAE

Una de las ventajas del espacio latente continuo del VAE es que podemos **interpolar** entre dos imágenes: tomar los vectores `mu` de dos imágenes reales, calcular puntos intermedios con una combinación lineal, y decodificarlos.

Si el espacio latente es continuo, los puntos intermedios deberían producir imágenes que mezclen suavemente las dos originales.

In [ ]:
def interpolate_latent(model, img_a, img_b, steps=10, device="cpu"):
    model.eval()
    with torch.no_grad():
        img_a = img_a.unsqueeze(0).to(device)
        img_b = img_b.unsqueeze(0).to(device)

        mu_a, _ = model.encoder(img_a)
        mu_b, _ = model.encoder(img_b)

        alphas = torch.linspace(0, 1, steps)
        images = []
        for alpha in alphas:
            z_interp = (1 - alpha) * mu_a + alpha * mu_b
            img = model.decoder(z_interp)
            images.append(img.squeeze(0))

    return images

def show_interpolation(model, dataset, class_a, class_b, steps=10, device="cpu"):
    idx_a = next(i for i, (_, l) in enumerate(dataset) if l == class_a)
    idx_b = next(i for i, (_, l) in enumerate(dataset) if l == class_b)

    img_a, _ = dataset[idx_a]
    img_b, _ = dataset[idx_b]

    interp_imgs = interpolate_latent(model, img_a, img_b, steps=steps, device=device)

    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.5, 2))
    for i, img in enumerate(interp_imgs):
        axes[i].imshow(img.cpu().squeeze(), cmap="gray")
        axes[i].axis("off")
        if i == 0:
            axes[i].set_title(class_names[class_a], fontsize=8)
        if i == steps - 1:
            axes[i].set_title(class_names[class_b], fontsize=8)

    plt.suptitle(f"Interpolacion: {class_names[class_a]} -> {class_names[class_b]}")
    plt.tight_layout()
    plt.show()

# Zapatilla -> Bota
show_interpolation(vae, test_dataset, class_a=7, class_b=9, steps=10, device=device)

# Remera -> Vestido
show_interpolation(vae, test_dataset, class_a=0, class_b=3, steps=10, device=device)

❓ Preguntas:

1. ¿La transición es suave o hay saltos abruptos? ¿A qué se debe?
2. ¿Qué pasa si intentás la misma interpolación usando el AE en lugar del VAE?
3. ¿Qué nos dice esto sobre la diferencia entre los espacios latentes de ambos modelos?

---

## 🧠 Parte 2: Para resolver

Al igual que en la Práctica 7, en esta segunda parte vamos a experimentar modificando decisiones clave del pipeline.

En particular, analizaremos:
- El tamaño del espacio latente.
- El parámetro beta del VAE.
- El desempeño en denoising.


### Ejercicio 1: Efecto del tamaño del espacio latente

Entrenás el AE con distintos valores de `latent_dim` y comparás:
- Calidad visual de las reconstrucciones.
- Estructura del espacio latente (t-SNE).

Probá al menos dos variantes: una con `latent_dim` más chico que el base (ej. 2 o 4) y una más grande (ej. 64 o 128).

¿Existe un trade-off entre compresión y calidad de reconstrucción?

In [ ]:
# Escribí tu código acá

### Ejercicio 2: Efecto del parámetro beta en el VAE

El parámetro `beta` controla el balance entre reconstrucción y regularización del espacio latente.

Entrenás el VAE con al menos tres valores de `beta` (ej. 0.1, 1.0, 5.0) y comparás:
- Calidad de las reconstrucciones.
- Estructura del espacio latente.
- Calidad de la interpolación entre clases.

¿Qué le pasa al espacio latente cuando `beta` es muy grande? ¿Y cuando es muy chico?

In [ ]:
# Escribí tu código acá

### Ejercicio 3: Denoising - AE vs VAE

Compará el desempeño de denoising entre el AE y el VAE para distintos niveles de ruido.

Usá `show_denoising` con ambos modelos para `noise_factor` en {0.2, 0.5, 0.8}.

¿Cuál funciona mejor en cada caso? ¿Por qué el VAE podría tener más o menos robustez al ruido que el AE?

In [ ]:
# Escribí tu código acá

---

## 📚 Recursos

- Python: https://docs.python.org/3/tutorial/
- NumPy: https://numpy.org/learn/
- Matplotlib: https://matplotlib.org/stable/index.html
- Google Colab: https://colab.research.google.com/

- PyTorch: https://docs.pytorch.org/tutorials/beginner/basics/intro.html
- Torchvision - datasets: https://docs.pytorch.org/vision/main/datasets.html
- Torchvision - transforms: https://docs.pytorch.org/vision/0.9/transforms.html
